# Test inverse metrics

testing if metrics of recovered signal match metrics of wavelet coefficients

In [ ]:
import sys
sys.path.append("..")
import numpy as np
import plotly.graph_objects as go


import wavelet_transform.wpt.transform as wpt
from wavelet_transform.wpt.inverse_transform import InverseTransform, inv_node
from wavelet_transform.utils.preprocessing import preprocess
from plotly.subplots import make_subplots

In [ ]:
sampling_rate = 200e6
frequency = 1.1e6
signal_len = 100e3

# Define test lengths as powers of base
t = np.arange(0, signal_len/sampling_rate, 1 / sampling_rate)
sine_wave = np.sin(2 * np.pi * frequency * t)  # 5 MHz sine wave
noise = np.random.normal(0, 1, sine_wave.shape) * np.std(sine_wave) * 0.1
signal = sine_wave #+ noise
signal = preprocess(signal, window=True)

In [ ]:
# fig = go.Figure()
# fig.add_trace(
#     go.Scatter(x=t, y=signal, mode="lines", name="Signal", line=dict(color="blue"))
# )

In [ ]:
# Parameters for the wavelet transform
wavelet = "db38"
levels = 8 # set to None to use max-level transform
mode = "constant"
best_basis = "energy"
min_level = 0
freq_threshold = min(1, 100e6 / (sampling_rate / 2)) # the nodes with norm_freq_range above this threshold will not be kept
fft_lim = 10  # this is the upper limit for plotting the spectrum (fft/psd of emissions)

keep_ratio = 1
inv_keep_ratio = keep_ratio

In [ ]:
transform = wpt.dwt(
    data=signal,
    wavelet=wavelet,
    max_level=levels,
    mode=mode,
    # best_basis=best_basis,
    # min_level=min_level,
    # freq_threshold=freq_threshold,
)
transform.keep_ratio = keep_ratio
wpt_df = transform.info(order="freq", leaf_nodes=True, keep_ratio=keep_ratio)

inv_tr_obj = InverseTransform(transform)
inv_df = inv_tr_obj.info(order="freq", leaf_nodes=True, keep_ratio=inv_keep_ratio)

In [ ]:
def normalise(array):
    return (array - np.min(array)) / (np.max(array) - np.min(array))

In [ ]:
metrics_to_plot = ["kurtosis", "energy", "spectral entropy"]
plot_vs = "level"  # "level" or "centre frequency"
x = wpt_df[plot_vs].to_numpy()
x = np.log(x) if plot_vs == "centre frequency" else x

# Define a mapping from levels to marker symbols
for metric_to_plot in metrics_to_plot:
    forward_metrics = wpt_df[metric_to_plot].to_numpy()
    backward_metrics = inv_df[metric_to_plot].to_numpy()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=x,
            y=forward_metrics,
            name="WPT " + metric_to_plot,
            mode="markers",
            marker=dict(symbol="circle", size=8)
        )
    )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=backward_metrics,
            name="Inverse WPT " + metric_to_plot,
            mode="markers",
            marker=dict(symbol="circle-open", size=8)
        )
    )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=forward_metrics/backward_metrics,
            name="Ratio WPT/Inv WPT " + metric_to_plot,
            mode="markers",
            marker=dict(symbol="diamond", size=8)
        )
    )

    fig.update_layout(
        title=f"Comparison of {metric_to_plot} between WPT and Inverse WPT",
        xaxis_title="centre frequency (Hz)" if plot_vs == "centre frequency" else "level",
        yaxis_title=metric_to_plot,
        barmode="group",
    )

    fig.show()



In [ ]:
nodes = transform.get_leaf_nodes(order="freq")
for i, node in enumerate(nodes):
    rec = inv_node(node)
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Wavelet Coeffs", "Reconstruction"))

    # use native indices for both traces
    x_node = np.arange(len(node.data))
    x_rec = np.arange(len(rec))

    fig.add_trace(
        go.Scatter(x=x_node, y=np.asarray(node.data), mode="lines", name="node data"),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=x_rec, y=np.asarray(rec), mode="lines", name="reconstructed data"),
        row=1, col=2
    )

    fig.update_xaxes(title_text="Sample index", row=1, col=1)
    fig.update_xaxes(title_text="Sample index", row=1, col=2)
    fig.update_yaxes(title_text="Amplitude", row=1, col=1)
    fig.update_yaxes(title_text="Amplitude", row=1, col=2)

    fig.update_layout(title_text=f"Node freq: {node.freq_range} and its reconstruction", showlegend=False)
    fig.show()

    for metric in metrics_to_plot:
        print(f"{metric} of node: {wpt_df.iloc[i][metric]}")
        print(f"{metric} of reconstruction: {inv_df.iloc[i][metric]}")
